# Day 080 — Exercise 1: Parsing the ReAct Format

**What you'll build:** `parse_react_step` — turns one `Thought / Action / Input` (or `Final Answer`) reply into a structured step.

**Why it matters:** ReAct's whole value is that the model *reasons before it acts*. But the reply is still just text — you have to parse the `Thought`, the `Action`, and the JSON `Input` out of it, and never crash on a malformed step.

In [ ]:
import json

def _make_mock_llm(script):
    """Return an llm_fn(messages) that yields each scripted reply in turn.

    Repeats the last reply once the script is exhausted - handy for testing a
    runaway loop (a model that never emits a Final Answer).
    """
    state = {'i': 0}
    def _fn(messages):
        i = state['i']
        state['i'] = min(i + 1, len(script) - 1)
        return script[i]
    return _fn
import ast
import json
import operator

# ── tools reused from Day 79: a safe calculator + a fact-lookup tool ──────────
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
    ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def _eval_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError("unsupported expression")


def safe_calculate(expression):
    """Evaluate arithmetic without eval() (see Day 79)."""
    return _eval_node(ast.parse(expression, mode="eval").body)


_FACTS = {
    "speed of light": "299792458 m/s",
    "pi": "3.14159",
    "earth radius": "6371 km",
    "days in a year": "365",
}


def _lookup(args):
    query = str(args.get("query", "")).lower().strip()
    for key, value in _FACTS.items():
        if query and (query in key or key in query):
            return value
    return "No result found for " + repr(args.get("query", ""))


DEFAULT_TOOLS = {
    "calculator": {
        "description": "Evaluate an arithmetic expression, e.g. 2 * (3 + 4).",
        "parameters": {"expression": "string - the arithmetic to evaluate"},
        "fn": lambda args: str(safe_calculate(args["expression"])),
    },
    "lookup": {
        "description": "Look up a known fact: speed of light, pi, earth radius, "
                       "days in a year.",
        "parameters": {"query": "string - what to look up"},
        "fn": _lookup,
    },
}


def build_tool_descriptions(tools):
    """Render a tool registry as prompt text (Day 79)."""
    lines = []
    for name, spec in tools.items():
        params = ", ".join(spec.get("parameters", {}))
        lines.append("- " + name + "(" + params + "): " + spec["description"])
    return "\n".join(lines)


def safe_parse_json(text):
    """Slice first '{' to last '}' and parse. Returns dict|None (Day 79)."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None


## Task

1. `_line_value(text, prefix)` — return the text after the first line that starts with `prefix` (case-insensitive), else `''`.
2. `_after_marker(text, marker)` — return everything after `marker` (case-insensitive), or `None` if it's absent.
3. `parse_react_step(text)` — extract the `Thought:`. If a `Final Answer:` marker is present → `{'type':'final','thought','answer'}`. Else if there's an `Action:` → `{'type':'action','thought','tool','input'}` (parse `Input:` with `safe_parse_json`, default `{}`). Otherwise fall back to a `final` action with the raw text. **Never raises.**

## Your Implementation

In [ ]:
def _line_value(text, prefix):
    """Text after the first line starting with prefix (case-insensitive)."""
    raise NotImplementedError

def _after_marker(text, marker):
    """Everything after marker (case-insensitive), or None if absent."""
    raise NotImplementedError

def parse_react_step(text):
    """Parse one ReAct step. NEVER raises.
    {'type':'action','thought','tool','input'} or {'type':'final','thought','answer'}.
    """
    raise NotImplementedError


In [ ]:

# ── parsing the ReAct format ──────────────────────────────────────────────────
def _line_value(text, prefix):
    """Text after the first line starting with prefix (case-insensitive), else ''."""
    for line in text.splitlines():
        if line.strip().lower().startswith(prefix.lower()):
            return line.strip()[len(prefix):].strip()
    return ""


def _after_marker(text, marker):
    """Everything after marker (case-insensitive), or None if absent."""
    idx = text.lower().find(marker.lower())
    if idx == -1:
        return None
    return text[idx + len(marker):].strip()


def parse_react_step(text):
    """Parse one ReAct step. NEVER raises.

    Returns either:
      {"type": "action", "thought": str, "tool": str, "input": dict}
      {"type": "final",  "thought": str, "answer": str}
    A reply with no recognisable Action falls back to a final answer holding
    the raw text - so a malformed step still ends the loop cleanly.
    """
    thought = _line_value(text, "Thought:")
    final = _after_marker(text, "Final Answer:")
    if final is not None:
        return {"type": "final", "thought": thought, "answer": final}
    action = _line_value(text, "Action:")
    if action:
        args = safe_parse_json(_line_value(text, "Input:")) or {}
        return {"type": "action", "thought": thought, "tool": action, "input": args}
    return {"type": "final", "thought": thought, "answer": text.strip()}


## Automated checks

In [ ]:

score, total = 0, 5
try:
    s = parse_react_step('Thought: I should add.\nAction: calculator\nInput: {"expression": "2+2"}')
    assert s['type'] == 'action' and s['tool'] == 'calculator'
    assert s['input']['expression'] == '2+2'
    score += 1; print("✅ parses a Thought/Action/Input step")

    assert 'add' in s['thought'].lower()
    score += 1; print("✅ extracts the Thought text")

    f = parse_react_step('Thought: done.\nFinal Answer: The result is 4.')
    assert f['type'] == 'final' and '4' in f['answer']
    score += 1; print("✅ parses a Final Answer step")

    g = parse_react_step('I have no idea, just rambling here.')
    assert g['type'] == 'final'
    score += 1; print("✅ falls back to final (never raises)")

    m = parse_react_step('thought: hmm\naction: lookup\ninput: {"query": "pi"}')
    assert m['type'] == 'action' and m['tool'] == 'lookup' and m['input']['query'] == 'pi'
    score += 1; print("✅ tolerant of lowercase markers")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── parsing the ReAct format ──────────────────────────────────────────────────
def _line_value(text, prefix):
    """Text after the first line starting with prefix (case-insensitive), else ''."""
    for line in text.splitlines():
        if line.strip().lower().startswith(prefix.lower()):
            return line.strip()[len(prefix):].strip()
    return ""


def _after_marker(text, marker):
    """Everything after marker (case-insensitive), or None if absent."""
    idx = text.lower().find(marker.lower())
    if idx == -1:
        return None
    return text[idx + len(marker):].strip()


def parse_react_step(text):
    """Parse one ReAct step. NEVER raises.

    Returns either:
      {"type": "action", "thought": str, "tool": str, "input": dict}
      {"type": "final",  "thought": str, "answer": str}
    A reply with no recognisable Action falls back to a final answer holding
    the raw text - so a malformed step still ends the loop cleanly.
    """
    thought = _line_value(text, "Thought:")
    final = _after_marker(text, "Final Answer:")
    if final is not None:
        return {"type": "final", "thought": thought, "answer": final}
    action = _line_value(text, "Action:")
    if action:
        args = safe_parse_json(_line_value(text, "Input:")) or {}
        return {"type": "action", "thought": thought, "tool": action, "input": args}
    return {"type": "final", "thought": thought, "answer": text.strip()}
```

**Why line-by-line instead of one big regex?** The ReAct format is line-oriented, so scanning lines for `Thought:` / `Action:` / `Input:` is simpler and more forgiving than a multiline regex — and it has no backslash or escaping traps.

</details>